## Prerequisistes

In [29]:
%load_ext autoreload
%autoreload 2

%pip install -qq --upgrade pip pandas
%pip install -qq numpy
%pip install -qq matplotlib
%pip install -qq seaborn
%pip install -qq jupyter
%pip install -qq qdrant-client
%pip install -qq sentence-transformers

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [30]:
import os
import pandas as pd
import numpy as np
from python import eda_utils as eda
from python import rag_utils as rag
from python import mistral_agent as mistral
import ipywidgets as widgets
from IPython.display import display

In [31]:
eda_selector = widgets.Dropdown(
    options=[True, False],
    value=False,
    description="EDA:"
)

display(eda_selector)

Dropdown(description='EDA:', index=1, options=(True, False), value=False)

In [32]:
rag_build = widgets.Dropdown(
    options=[True, False],
    value=False,
    description="RAG Build:"
)

display(rag_build)

Dropdown(description='RAG Build:', index=1, options=(True, False), value=False)

In [33]:
eda_enabled = eda_selector.value
rag_build_enabled = rag_build.value
print("Configurație: rag_build_enabled =", rag_build_enabled, "| eda_enabled =", eda_enabled)

Configurație: rag_build_enabled = False | eda_enabled = False


In [ ]:
DATASET_DIR = "dataset/"
TEMPLATE_DIR = "template/"
QDRANT_URL = "http://localhost:40081"
OLLAMA_URL = "http://localhost:40082/api/generate"
MODEL_NAME = "romistral"

## Exploratory Data Analysis

In [35]:
if eda_enabled:
    df = eda.load_data(DATASET_DIR + 'products.csv')
    df = eda.clean_column_names(df)
    df = eda.drop_empty_columns(df)
else:
    df = pd.read_csv(DATASET_DIR + 'products_clean.csv')

In [36]:
if eda_enabled:
    df = eda.aggregate_by_handle(df, separator=" ; ")
    df = df.drop(columns=['Handle'])
    df = eda.compress_variations(df, column_name='Variations')

In [37]:
if eda_enabled:
    eda.display_low_cardinality_info(df, threshold=1)

In [38]:
if eda_enabled:
    df = eda.drop_low_cardinality_columns(df, threshold=1)

In [39]:
if eda_enabled:
    eda.display_low_cardinality_info(df, threshold=4)

In [40]:
if eda_enabled:
    df = df.drop(columns=['Published', 'Option1 Name', 'Option2 Name', 'Variant Taxable', 'Complementary Heading',  'Product rating count', 'Dietary preferences', 'Plant characteristics'])
    df = df.drop(columns=['Status', 'Image Src', 'Image Position', 'Variant SKU'])

In [41]:
if eda_enabled:
    eda.display_low_cardinality_info(df, threshold=4)

In [42]:
if eda_enabled:
    eda.display_missing_info(df, min_threshold=0.85)

In [43]:
if eda_enabled:
    df = df.drop(columns=['Cost per item', 'Shape', 'Variant Image'])

In [44]:
if eda_enabled:
    eda.analyze_column(df, 'Option2 Value', top_n=30)
    eda.analyze_column(df, 'Tags', top_n=30)
    eda.analyze_column(df, 'Option1 Value', top_n=30)
    eda.analyze_column(df, 'Variant Compare At Price', top_n=30)
    eda.analyze_column(df, 'Google Shopping / Google Product Category', top_n=30)
    eda.analyze_column(df, 'Badges', top_n=30)
    eda.analyze_column(df, 'Custom Label #1', top_n=30)
    eda.analyze_column(df, 'Custom Label #2', top_n=30)
    eda.analyze_column(df, 'Custom Label #3', top_n=30)
    eda.analyze_column(df, 'Custom Label #4', top_n=30)
    eda.analyze_column(df, 'Custom Label #5', top_n=30)
    eda.analyze_column(df, 'Floral Arrangements Type', top_n=30)
    eda.analyze_column(df, 'Variations', top_n=30)
    eda.analyze_column(df, 'Allergen information', top_n=30)

In [45]:
if eda_enabled:
    df = df.drop(columns=['Option2 Value', 'Variant Compare At Price', 'Google Shopping / Google Product Category', 'Custom Label #3'])

In [46]:
if eda_enabled:
    df['Option1 Value'] = df['Option1 Value'].replace('Default Title', np.nan)
    df = df.rename(columns={
        'Option1 Value': 'Details',
        'Badges': 'Tags2',
        'Custom Label #1': 'Sezon',
        'Custom Label #2': 'Parfum',
        'Custom Label #3': 'Polen',
        'Custom Label #4': 'Cadou',
        'Custom Label #5': 'Dimensiune'
    })
    df['Dimensiune'] = df['Dimensiune'].str.replace('Dimensiune ', '', regex=False).str.strip()

In [47]:
if eda_enabled:
    eda.display_missing_info(df, min_threshold=0.85)

In [48]:
if eda_enabled:
    eda.display_missing_info(df, min_threshold=0.5, max_threshold=0.85)

In [49]:
if eda_enabled:
    eda.display_missing_info(df, min_threshold=0.25, max_threshold=0.5)

In [50]:
if eda_enabled:
    df = eda.clean_html_column(df, column_name='Body (HTML)')

In [51]:
if eda_enabled:
    for col in df.columns:
        eda.analyze_column(df, col, top_n=30)

In [52]:
if eda_enabled:
    print(f"\n✨ FINAL DATASET: {df.shape[0]} Products, {df.shape[1]} Columns ✨")

    file_to_save = DATASET_DIR + 'products_clean.csv'
    if os.path.exists(file_to_save):
        os.remove(file_to_save)
        print(f"Deleted old version of {file_to_save}")

    df.to_csv(file_to_save, index=False)
    print(f"Successfully saved new {file_to_save}")

In [53]:
if eda_enabled:
    docs_pentru_qdrant = eda.export_for_qdrant(df, output_json=TEMPLATE_DIR + 'qdrant_payload.json')

## RAG

In [54]:
if rag_build_enabled:
    rag.build_and_index_rag(df, output_json=TEMPLATE_DIR + 'qdrant_payload.json', qdrant_url=QDRANT_URL)

In [55]:
# ==========================================
# CELULA 2: TESTĂM REȚEAUA SOTA (Hibrid + Reranker)
# ==========================================
# Aici simulezi ce va extrage Mistral Router-ul:
simulated_state = {
  "filters": {
    "price_max": 400,
    "category": "Buchete Flori"
  },
  "semantic_query": "Vreau un buchet cu lalele și un aspect elegant, de primăvară"
}

# Aici testam functia SOTA care cauta si aplica cross-encoderul
rezultate_sota = rag.sota_hybrid_search(simulated_state, final_top_k=3, qdrant_url=QDRANT_URL)

# Printam contextul asa cum il va vedea modelul RoMistral final
print(rag.format_context_for_mistral(rezultate_sota))

🔌 Ne conectăm la Qdrant (http://localhost:40081)...


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


PRODUSE RECOMANDATE (din baza de date):
1. Buchet de Flori "Spring Texture" | Preț: 292.0 RON
Detalii: Produs: Buchet de Flori "Spring Texture".
Type: Buchete Flori
Flowers Type: Lalele
Colors: Pastel
Dimensiune: Mica
Event Type: 1 Martie
8 Martie
Event Corporate
Aniversare
Cadou Mama/Bunica
Body (HTML): Descopera frumusetea complexa a primaverii cu buchetul "Spring Texture". Am selectat 15 lalele duble (tip bujor) , cunoscute pentru numarul mare de petale si volumul impresionant pe care il capata atunci cand se deschid. Buchetul este un joc vesel de culori, combinand energia lalelelor galbene cu eleganta celor mov/roz-purpuriu si roz pal. Pentru a accentua senzatia de catifelare, am inserat fire de Lagurus (coada iepurelui) â€“ acele elemente decorative pufoase care adauga un farmec aparte compozitiei. Designul este completat de un ambalaj premium: o hartie alba plisata (texturata) , care ofera rigiditate si un aspect modern, curat. Buchetul este legat cu panglica personalizata City F

In [56]:
# Mesajul de la cumpărător
mesaj_client = "Aș vrea un buchet elegant pentru o aniversare, cu lalele albe. Bugetul e undeva pe la 350 lei."

print(f"💬 Clientul a spus: '{mesaj_client}'\n" + "-"*50)

# ==========================================
# 2. FLUXUL MACRO (Orchestrarea)
# ==========================================

# PASUL A: Router-ul gândește și creează filtrele (JSON)
print("⚙️ [MISTRAL] Extrag filtre...")
rag_state = mistral.extract_filters(
    user_message=mesaj_client,
    ollama_url=OLLAMA_URL,
    model_name=MODEL_NAME,
    template_dir=TEMPLATE_DIR
)

# PASUL B: RAG caută și ordonează produsele din baza de date
print("🔍 [QDRANT+RERANKER] Caut în catalog...")
rezultate_sota = rag.sota_hybrid_search(
    state_json=rag_state, 
    final_top_k=3, 
    qdrant_url=QDRANT_URL
)
context_produse = rag.format_context_for_mistral(rezultate_sota)

# PASUL C: Agentul de Vânzări răspunde frumos
print("✍️ [MISTRAL] Generez răspunsul final...\n" + "-"*50)
raspuns_final = mistral.generate_conversational_response(
    user_message=mesaj_client,
    rag_context=context_produse,
    ollama_url=OLLAMA_URL,
    model_name=MODEL_NAME
)

print(raspuns_final)

💬 Clientul a spus: 'Aș vrea un buchet elegant pentru o aniversare, cu lalele albe. Bugetul e undeva pe la 350 lei.'
--------------------------------------------------
⚙️ [MISTRAL] Extrag filtre...
⚠️ Eroare la Mistral Router: 404 Client Error: Not Found for url: http://localhost:40082/api/generate
🔍 [QDRANT+RERANKER] Caut în catalog...
🔌 Ne conectăm la Qdrant (http://localhost:40081)...


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


✍️ [MISTRAL] Generez răspunsul final...
--------------------------------------------------
Îmi cer scuze, întâmpinăm o problemă tehnică de comunicare. Vă rugăm să reîncercați. (Detalii eroare: 404 Client Error: Not Found for url: http://localhost:40082/api/generate)
